# Set 05 – XAI mit Modellkoeffizienten

Explainable AI, kurz XAI, versucht nachvollziehbar zu machen, wie ein Modell zu seinen Ergebnissen kommt. Lineare und logistische Regression sind dafür besonders geeignet: Ihre Koeffizienten zeigen Richtung und Stärke des modellierten Zusammenhangs.

Wichtig: Ein großer Koeffizient beweist keine Ursache. Er beschreibt nur das Verhalten dieses Modells auf diesen Daten.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import r2_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(42)

## Teil A – Koeffizienten einer linearen Regression

Wir erzeugen kontrollierte Maschinendaten. Der Energieverbrauch hängt positiv von Laufzeit, Auslastung und Wartungsalter sowie negativ von der Außentemperatur ab.

In [ ]:
n = 500
regression_daten = pd.DataFrame({
    "laufzeit_stunden": rng.uniform(4, 20, n),
    "auslastung_prozent": rng.uniform(25, 100, n),
    "aussentemperatur_c": rng.uniform(-5, 35, n),
    "wartungsalter_monate": rng.uniform(0, 48, n),
})
regression_daten["energie_kwh"] = (
    120
    + 8.0 * regression_daten["laufzeit_stunden"]
    + 3.5 * regression_daten["auslastung_prozent"]
    - 4.5 * regression_daten["aussentemperatur_c"]
    + 1.8 * regression_daten["wartungsalter_monate"]
    + rng.normal(0, 28, n)
)
display(regression_daten.head())

## 1. Koeffizienten in Originaleinheiten

Ohne Skalierung lässt sich jeder Koeffizient in der Einheit des jeweiligen Merkmals lesen. Beispielsweise bedeutet der Koeffizient der Laufzeit: erwartete Änderung der Kilowattstunden bei einer zusätzlichen Stunde, wenn die anderen Merkmale gleich bleiben.

Die Beträge dürfen hier nicht direkt als Wichtigkeitsranking verglichen werden, weil Stunden, Prozent und Grad unterschiedliche Skalen besitzen.

In [ ]:
X_reg = regression_daten.drop(columns="energie_kwh")
y_reg = regression_daten["energie_kwh"]
X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=42
)

regression_roh = LinearRegression()
regression_roh.fit(X_reg_train, y_reg_train)

roh_koeffizienten = pd.DataFrame({
    "Merkmal": X_reg.columns,
    "Koeffizient_Originaleinheit": regression_roh.coef_,
})
display(roh_koeffizienten.round(3))
print("Test-R²:", round(r2_score(y_reg_test, regression_roh.predict(X_reg_test)), 3))

## 2. Standardisierte Koeffizienten vergleichen

Der StandardScaler bringt alle Eingangsmerkmale auf eine vergleichbare Skala. Danach beschreibt jeder Koeffizient die Änderung der Vorhersage bei einer Erhöhung um eine Standardabweichung.

- Das Vorzeichen zeigt die Richtung.
- Der Betrag dient als modellinterne Wichtigkeit.

In [ ]:
regression_pipeline = Pipeline([
    ("skalierung", StandardScaler()),
    ("modell", LinearRegression()),
])
regression_pipeline.fit(X_reg_train, y_reg_train)

reg_modell = regression_pipeline.named_steps["modell"]
reg_xai = pd.DataFrame({
    "Merkmal": X_reg.columns,
    "Koeffizient": reg_modell.coef_,
})
reg_xai["absolute_Wichtigkeit"] = reg_xai["Koeffizient"].abs()
reg_xai = reg_xai.sort_values("Koeffizient")
display(reg_xai.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
farben = np.where(reg_xai["Koeffizient"] >= 0, "#54A24B", "#E45756")
ax.barh(reg_xai["Merkmal"], reg_xai["Koeffizient"], color=farben)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("standardisierter Koeffizient")
ax.set_title("Globale Koeffizienten der linearen Regression")
plt.show()

## Teil B – Koeffizienten einer logistischen Regression

Nun erzeugen wir Sensordaten. Klasse 1 steht für einen auffälligen Zustand. Die logistische Regression modelliert Log-Odds; positive Koeffizienten erhöhen die modellierte Wahrscheinlichkeit für Klasse 1, negative senken sie.

In [ ]:
n = 700
klassifikation_daten = pd.DataFrame({
    "temperatur_c": rng.normal(70, 9, n),
    "vibration_mm_s": rng.normal(4.0, 1.2, n),
    "druck_bar": rng.normal(8.0, 1.0, n),
    "alter_monate": rng.uniform(0, 72, n),
})

score = (
    0.75 * (klassifikation_daten["temperatur_c"] - 70) / 9
    + 1.5 * (klassifikation_daten["vibration_mm_s"] - 4.0) / 1.2
    - 0.8 * (klassifikation_daten["druck_bar"] - 8.0)
    + 0.55 * (klassifikation_daten["alter_monate"] - 36) / 20
    - 1.0
)
wahrscheinlichkeit = 1 / (1 + np.exp(-score))
klassifikation_daten["auffaellig"] = rng.binomial(1, wahrscheinlichkeit)

X_klass = klassifikation_daten.drop(columns="auffaellig")
y_klass = klassifikation_daten["auffaellig"]
X_klass_train, X_klass_test, y_klass_train, y_klass_test = train_test_split(
    X_klass, y_klass, test_size=0.25, random_state=42, stratify=y_klass
)
print(y_klass.value_counts(normalize=True).round(3))

## 3. Logistische Pipeline und Koeffizienten

Durch die Skalierung bezieht sich jeder Koeffizient auf eine Standardabweichung. exp(Koeffizient) ist das zugehörige Odds Ratio.

Ein Odds Ratio über 1 erhöht die Odds für Klasse 1, ein Wert unter 1 senkt sie. Odds sind nicht dasselbe wie Wahrscheinlichkeit.

In [ ]:
klass_pipeline = Pipeline([
    ("skalierung", StandardScaler()),
    ("modell", LogisticRegression()),
])
klass_pipeline.fit(X_klass_train, y_klass_train)

klass_modell = klass_pipeline.named_steps["modell"]
klass_xai = pd.DataFrame({
    "Merkmal": X_klass.columns,
    "Koeffizient": klass_modell.coef_[0],
})
klass_xai["absolute_Wichtigkeit"] = klass_xai["Koeffizient"].abs()
klass_xai["Odds_Ratio"] = np.exp(klass_xai["Koeffizient"])
klass_xai = klass_xai.sort_values("Koeffizient")

display(klass_xai.round(3))
print("Test-Recall:", round(recall_score(y_klass_test, klass_pipeline.predict(X_klass_test)), 3))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

farben = np.where(klass_xai["Koeffizient"] >= 0, "#54A24B", "#E45756")
axes[0].barh(klass_xai["Merkmal"], klass_xai["Koeffizient"], color=farben)
axes[0].axvline(0, color="black", linewidth=1)
axes[0].set_title("Richtung und Stärke")
axes[0].set_xlabel("standardisierter Koeffizient")

wichtigkeit = klass_xai.sort_values("absolute_Wichtigkeit")
axes[1].barh(wichtigkeit["Merkmal"], wichtigkeit["absolute_Wichtigkeit"], color="#4C78A8")
axes[1].set_title("Absolute Modellwichtigkeit")
axes[1].set_xlabel("Betrag des Koeffizienten")

plt.tight_layout()
plt.show()

## 4. Lokale Erklärung einer einzelnen Vorhersage

Für einen einzelnen Fall kann der lineare Score als Summe der Beiträge berechnet werden:

Beitrag eines Merkmals = standardisierter Merkmalswert mal Koeffizient

Positive Beiträge schieben die Vorhersage in Richtung Klasse 1, negative in Richtung Klasse 0.

In [ ]:
fall = X_klass_test.iloc[[0]]
fall_skaliert = klass_pipeline.named_steps["skalierung"].transform(fall)[0]
beitraege = fall_skaliert * klass_modell.coef_[0]

lokale_xai = pd.DataFrame({
    "Merkmal": X_klass.columns,
    "Rohwert": fall.iloc[0].values,
    "standardisierter_Wert": fall_skaliert,
    "Koeffizient": klass_modell.coef_[0],
    "Beitrag_zum_Score": beitraege,
}).sort_values("Beitrag_zum_Score")

display(lokale_xai.round(3))
print("Intercept:", round(klass_modell.intercept_[0], 3))
print("Summe Score:", round(klass_modell.intercept_[0] + beitraege.sum(), 3))
print("P(Klasse 1):", round(klass_pipeline.predict_proba(fall)[0, 1], 3))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
farben = np.where(lokale_xai["Beitrag_zum_Score"] >= 0, "#54A24B", "#E45756")
ax.barh(lokale_xai["Merkmal"], lokale_xai["Beitrag_zum_Score"], color=farben)
ax.axvline(0, color="black", linewidth=1)
ax.set_xlabel("Beitrag zum linearen Score")
ax.set_title("Lokale Erklärung eines Testfalls")
plt.show()

## Grenzen dieser XAI-Methode

1. Koeffizienten erklären das Modell, nicht automatisch die reale Welt.
2. Ohne Skalierung sind Beträge verschieden skalierter Merkmale nicht vergleichbar.
3. Stark korrelierte Merkmale können sich Wichtigkeit teilen oder instabile Koeffizienten erzeugen.
4. Ein global wichtiger Faktor muss nicht für jeden Einzelfall entscheidend sein.
5. Ein positiver Koeffizient beweist keine Kausalität.
6. Bei nichtlinearen Modellen sind Koeffizienten nicht in derselben Form verfügbar.

XAI sollte deshalb mit Datenanalyse, Fachwissen und Prüfung der Modellgüte verbunden werden.